## Upravljanje z omrežji

- Manjša vezja so se povezala v VELIKO inftastrukturo
- SISTEMATIČNO moramo upravljati strojne in programske komponente!

Definicija: Upravljanje z omrežjem vključuje **vpeljavo**, **integracijo** in
**koordinacijo** s *strojno opremo*, *programsko opremo* in *človeškimi viri* z namenom **opazovanja, testiranja, konfiguriranja, analiziranja in nadzorovanja omrežnih virov**, pri katerih želimo zagotoviti delovanje v realnem času (ali delovanje z ustrezno kakovostjo – QoS) za sprejemljivo
ceno.

### Trije osnovni gradniki
1) **Management informacij** - (SNMP, MIB, podatkovni modeli)
- Skrbi za to, katere informacije o omrežju zbiramo in hranimo

2) **Management protokoli** - (SNMP, NetConf, RESTCONF …)
- Skrbijo za to, kako prenašamo in izmenjujemo te informacije med managerji in agenti.

3) **Management aplikacije / orodja** - (NMS sistemi, monitoring sistemi – npr. Zabbix, Nagios)
- Uporabljajo informacije in protokole za nadzor, konfiguracijo, alarmiranje itd.

## Področja vpravljanja:
- Upravljanje z **Napakami**
- Upravljanje z **Konfiguracijami**
- Upravljanje z **Varnostjo**
- Upravljanje z **Beležejem dostopom**

### Aktivnosti upravljanja
1. **Zaznavanje napake** na vmesniku računalnika ali usmerjevalnika:
- *programska oprema lahko sporoči administratorju, da je na vmesniku prišlo do težave (celo preden odpove!)*
2. **Nadzorovanje delovanja** računalnikov in **analiza** omrežja
3. **Nadzorovanje omrežnega prometa**:
- *administrator lahko opazuje pogoste smeri komunikacij in najde ozka grla*
4. **Zaznavanje hitrih sprememb** v usmerjevalnih tabelah:
- *ta pojav lahko opozarja na težave z usmerjanjem ali napako v usmerjevalniku*
5. **Nadzorovanje nivoja zagotavljanja storitev**:
- *ponudniki omrežnih storitev nam lahko jamčijo razpoložljivost, zanasnitev in določeno prepustnost storitev; administrator lahko meri in preverja*
6. **Zaznavanje vdorov**:
- *administrator je lahko obveščen, če določen promet prispe iz sumljivih virov*
- *zaznava lahko tudi določen tip prometa (npr. množica SYN paketov, namenjena enem samem vmesniku)*

## Viri v omrežju:
- **Strojni viri** – npr. tiskalniki, strežniki, diski
- **Programski viri** – aplikacije, baze, licenčni strežniki
- **Podatkovni viri** – datoteke, podatkovne baze, backupi
- **Komunikacijski viri** – pasovna širina, IP-ji, DNS, DHCP
- **Varnostni viri** – certifikati, ACL, požarni zidovi
- **Virtualni/oblačni viri** – VM, VLAN, cloud storitve

## MIB moduli
Prenos podatkov/storitev -> [SNMP]:
- Management Information Base z upravljanimi objekti in njihovimi parametri
[MBI-Poimenovanje,Podatkov](./Slike/MIB_poimenovanje.png) (**1.3.6.1.2.1.7 določa protokol UDP**)

**MDB**: Baza z temi podatki za vsako napravo
**SMI**: Jezik za dešifracijo teh podatkov

### SMI:
- Vsebinsko povezuje objekte
Definicija objekta:
- **ima podatkovni tip**,
- **status**,
- **opis pomena**

#### Ostalo:

- IETF (Internet Engineering Task Force) zadolžena za standardizacijo MIB modulov za usmerjevalnike, vmesnike in drugo omrežno opremo (Poimenovanje)
- osnovni podatkovni tipi: *INTEGER, Integer32,Unsigned32, OCTET STRING, OBJECT IDENTIFIED, IPaddress, Counter32, Counter64, Gauge32, Time Ticks, Opaque*
- sestavljeni podatkovni tipi: *OBJECT-TYPE, MODULE-TYPE*

## SNMP - Simple Network Management Protokol    
SNMP uporablja za prenos podatkov format Type Length Value (TLV) oziroma bolj specifično ASN.1 BER kodiranje.   
- **NI NAMENJEN ZA APLIKACIJE** (rešitev: NETCONF z YANG)

    
- protokol za izmenjavo nadzornih informacij med upravljalcem in nadzorovanimi objekt   
V paketu:
- Indetifikator seje
- Tip sporočila
- Pošli/Prejmi
- Verzija SNMP-ja   
[Oblika_sporočila](./Slike/SNMP_Sporočilo.png)  
[Oblika_sporočila_zahteva/odgovor](./Slike/SNMP_zahteva_odgovor.png)     *PUD type*     
[Oblika_obvestila](./Slike/SNMP_Obvestilo.png)  

### Delovanje:  
[Delovanje_SNMP_protokola](./Slike/SNMP_delovanje.png)  
Nadzor: vprašanje -> Naprava: odgovor   
Naprava: sporočilo -> Nadzor: potrditev 
- Ukazi: (tipi sporočil)
    - GetRequest: upravljalec -> agent
    - SetRequest: upravljalec -> agent
    - Response: agent -> upravljalec
    - Trap: agent -> upravljalec
    - InformRequest: upravljalec -> upravljalec

- Če naprava pošlje odgovor, nima pojma ali ga je dobila. (UDP)
- Če nadzorni sistem ne dobi podatkov še enkrat povpraša.
    - Saj v tem primiru napreve ne zanima ali je nadrorni sistem dobil podatke (**Nadzorni sistem želi te podatke**) (za vprašanje-odgovor)
- Drugače pa ravno obratno (za sporočilo-potrditev)
- **V primeru neuspešnosti ponovimo**   

Zato določimo indetifikator seje ("zaporedno št.") valja za obe smeri:  
- **To je zato da povežemo vprašanje in odgovor in sporočilo in potrditev**
- Da vemo katere podatke, kdo bere in pod kakšnimi varnosnimi pogoji
- Ni dejansko saja vendar je samo dogovor o podatkih med MIB in SNMP

### SNMP Problemi-Rešitve:  
1. **Velikost paketov**
    - Problem: SNMP paketi lahko vsebujejo veliko podatkov (MIB objekti), UDP pa ima omejeno velikost segmenta (~64 KB, v praksi manj).
    - Rešitev: SNMP uporablja fragmentacijo paketov ali GETBULK za branje več objektov po delih.
    - Omeji velikost prenosa, da ustreza UDP segmentu.
2. **Ponovno pošiljanje (retransmission)**
    - Problem: UDP je brez potrditev, zato ni zagotovljene dostave.
    - Rešitev: SNMP manager sam ponovno pošlje zahtevo, če ne prejme odgovora v določenem času.
    - Tako je nadzor dostave implementiran na višjem OSI nivoju (**aplikacijskem**).
3. **Izgubljena obvestila (traps)**
    - Problem: če se TRAP sporočilo izgubi, pošiljatelj (agent) ne ve, prejemnik (manager) pa ga ne prejme.
    - Rešitev: 
        - SNMPv1/v2c: TRAP ni potrjen, ni zagotovila.
        - SNMPv2/3: uvedeni INFORM sporočila, ki zahtevajo potrditev od prejemnika → izgubljeni trap se lahko ponovno pošlje.

## SNMPv3   
**v3 šele zasnovana za varnost**    
Prekrivanje (Omogočamo celovitost podatkov) 
- Spečificirati moramo tudi ali je danki ali debeli konec (zato potrebujemo)
Primer: BER 
- Jezik opisa prenešenih podatkov:
    - Jason
    - XML
    - Ker je bilo to zelo komplicirano so vzpostavili ASN.1 standard, ki nam pove kako so zapisani prenešeni podatki.

## SNMP varnost 
Če imamo varnostne mehanizme in rešeno ponovno pošiljanje "je to to".   
    
### Varnostni mehanizmi:    
Ti paketi vsi preverjajo oz. izračunavajo neko vrednost, ki se ob spremembi bita spremeni   
- Paritetna vsota (prešteje vse bite ali besede in jih sešteje modularno 2^n)
- CRC (Matematična funkcija, ki izračuna kontrolno kodo za blok podatkov)
- Paritetni bit (En bit nam pove ali je število bitov sodo ali liho)    
To je strukturiran SNMP paket (Tip sporocila, Indetivikator seje, Podatke)  
- PDU (Šifriranje vsebine paketov) (ASN.1 definira standarde kodiranja)(?Predstavitev storitev?)

## SNMPv3 – Varnostni mehanizmi

### 1. Šifriranje (privacy)
- Šifrirana je vsebina PDU.
- Uporablja se DES (CBC način).
- Ključi se morajo vnaprej izmenjati (USM ne določa avtomatske izmenjave ključev).

---

### 2. Celovitost (integrity)
- Uporablja se HMAC (npr. HMAC-MD5 ali HMAC-SHA1).
- Prejemnik preveri HMAC, da zazna:
  - spremembe podatkov
  - ponarejena ali vstavljena sporočila

---

### 3. Zaščita proti ponovitvi (anti-replay)
SNMPv3 uporablja **žeton**, ki ga sestavljata:

- `engineBoots` – število vseh ponovnih zagonov SNMP agenta  
- `engineTime` – čas, ki teče od zadnjega zagona

Ti dve vrednosti delujeta kot **anti-replay token**, ker mora biti vsak prejet paket:
- časovno svež,
- z ustreznim `engineBoots`/`engineTime`,
- drugače agent paket zavrne.

Možna je tudi uporaba **nonce** vrednosti (enkratne številke), odvisno od implementacije.

---

### 4. Nadzor dostopa (access control)
- Temelji na uporabniških imenih in varnostnih nivojih (noAuthNoPriv, authNoPriv, authPriv).
- Pravice določajo, kateri uporabniki lahko:
  - berejo OID-e,
  - nastavljajo vrednosti.
- Konfiguracija se hrani v **Local Configuration DataStore (LCD)**, ki je sam predstavljen kot SNMP objekt.



## Predstavitev storitev
1. Pošiljatelj upošteva obliko podatkov, ki jo uporablja prejemnik:
- podatke pretvarja v njegovo obliko in nato šele pošlje.
2. Pošiljatelj pošlje podatke v svoji obliki, prejemnik pretvori v lastno obliko.
3. Pošiljatelj pretvori v neodvisno obliko in nato pošlje. 
- Prejemnik neodvisno obliko pretvori v svojo lastno obliko.

### ASN.1 (3. Neodvisno oblika)
[BER-TLV](./Slike/BER-TLV.png)
- Pri zapisovanju tipov se uporablja pravila **BER (Binary Encoding Rules)**.
- Ta definirajo zapis podatkov po principu **TLV (T=Type, L=Length, V=Value)**.

## NETCONF in YANG

### NETCONF (Network Configuration Protocol)
- je protokol za upravljanje in konfiguracijo omrežnih naprav.   

- Ključne lastnosti:    
    - Uporablja SSH kot transport (varno!).
    - Uporablja XML za podatke.
    - Deluje z RPC (Remote Procedure Call) klici.
    - Podpira transakcije → spremembe so atomske (all-or-nothing).
    - Omogoča get-config, edit-config, copy-config, delete-config.
    - Omogoča lock/unlock konfiguracije (prepreči konflikte).
    - Validacija konfiguracije pred uveljavitvijo.
    - Dela s running, startup in candidate konfiguracijo.

- Osnovne operacije:
    - get-config → pridobi konfiguracijo
    - get → statusni podatki
    - edit-config → spremeni konfiguracijo
    - commit → potrdi spremembe
    - lock/unlock → zaklene konfiguracijo

- Zakaj NETCONF?
    - Standardiziran, varen, avtomatiziran način konfiguracije.
    - Nadomešča nepredvidljivo CLI skriptiranje.

### YANG (Yet Another Next Generation)
- je modelirni jezik, ki opisuje:

- konfiguracijske podatke,

- statusne podatke,

- RPC,

- notifikacije.

- Ključne lastnosti:
    - Je data modeling language – ne konfigurira, ampak opisuje modele.
    - Modeli so organizirani kot drevo (hierarhija).
    - Definira tipe podatkov, omejitve, default vrednosti.
    - Uporabljajo ga NETCONF, RESTCONF in moderni SDN sistemi.

- Osnovni gradniki YANG:
    - module – glavni dokument
    - container – logična skupina
    - leaf – preprosto polje
    - list – tabelarni podatki
    - type – int, string, boolean, enum...
    - must in range – validacija
    - default – privzeta vrednost

- Primer uporabe:
    - YANG definira strukturo → NETCONF jo uporablja pri branju/urejanju konfiguracije.

### Vloga aplikacije
- Aplikacija komunicira z upravljalcem, ne neposredno z napravami.

- Controller uporablja NETCONF/RESTCONF, YANG modele in RPC-je.

- YANG določi podatkovni model → controller ga uporabi za validacijo.

- NETCONF poskrbi za varno, atomsko spremembo konfiguracije.

- Aplikacija deli ukaze v obliki API klicev (običajno JSON).

- Aplikacija ne bere neposredno konfiguracije, ampak:
    - pošlje zahtevo controllerju (npr. “ustvari nov VLAN”, “spremeni IP”)

- controller pretvori to zahtevo v:
    - NETCONF RPC
    - ali RESTCONF REST klic
    - ob upoštevanju YANG modelov

- Aplikacija ne rabi poznati XML ali NETCONF — samo API controllerja.